# 07. Split-Apply-Combine Groupby Operations: Beginner Guide

### 🌟 What is the Split-Apply-Combine Pattern (`.groupby()`)?
The **Split-Apply-Combine** pattern is how Pandas groups data by categories (e.g. by region or card type), applies aggregation functions (`sum`, `mean`, `count`, custom functions), and combines the results into a clean summarized table.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Grouping**: Covers `df.groupby()`.
- **Multiple Aggregations**: Covers `.agg()` with multiple metrics.
- **Named Aggregations**: Covers named aggregation tuples (`total_amt=('transaction_amount', 'sum')`).
- **Group Transformation**: Covers `.transform()`.
- **Group Filtering**: Covers `.filter()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount   card_type  \
0       TX110686      C82845       M2697             1216.33        Visa   
1       TX107170      C85674       M3868              324.99  MasterCard   

  transaction_status device_type  account_age_months transaction_date region  \
0             Failed         POS                  56       07-06-2025  North   
1           Reversed     Desktop                 112       10/05/2025   East   

   is_fraud  
0         0  
1         0  


### 🔹 Grouping with `df.groupby()`
Partitions transactions by region and card_type. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** After running `.groupby().agg()`, the grouped columns become index levels. Call `.reset_index()` to bring them back as standard columns.

**Syntax:** `df.groupby(['region', 'card_type'])`


In [2]:
grouped_region = df.groupby('region')

### 🔹 Multi-Metric Aggregation with `.agg()`
Computes total spend, average spend, and fraud rate per region. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.groupby('region').agg({'transaction_amount': ['sum', 'mean'], 'is_fraud': 'mean'})`


In [3]:
regional_summary = df.groupby('region').agg({
    'transaction_amount': ['sum', 'mean', 'count'],
    'is_fraud': 'mean'
})
print('Regional Multi-Metric Summary:\n', regional_summary)

Regional Multi-Metric Summary:
         transaction_amount                     is_fraud
                       sum         mean count      mean
region                                                 
 East             66059.06   930.409296    71  0.052632
 North            89344.88  1103.023210    81  0.097561
 South            86816.88  1045.986506    83  0.105882
 West             68143.46   908.579467    75  0.077922
East            3450077.53   995.980811  3464  0.098772
North           3494357.16  1017.576342  3434  0.112715
South           3370644.64  1002.571279  3362  0.104773
West            3394291.07   998.320903  3400  0.105750
east             102348.29  1149.980787    89  0.173913
north             66575.98  1091.409508    61  0.169231
south             87417.99  1150.236711    76  0.111111
west              73461.20  1113.048485    66  0.101449


### 🔹 Named Aggregations Syntax
Generates clean single-level column names for aggregated transaction metrics. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.groupby('region').agg(total_spend=('transaction_amount', 'sum'), avg_spend=('transaction_amount', 'mean'), fraud_rate=('is_fraud', 'mean'))`


In [4]:
clean_named_agg = df.groupby('region').agg(
    total_spend=('transaction_amount', 'sum'),
    avg_spend=('transaction_amount', 'mean'),
    fraud_rate=('is_fraud', 'mean'),
    tx_count=('transaction_id', 'count')
)
print('Clean Named Aggregations Table:\n', clean_named_agg.round(3))

Clean Named Aggregations Table:
          total_spend  avg_spend  fraud_rate  tx_count
region                                               
 East       66059.06    930.409       0.053        76
 North      89344.88   1103.023       0.098        82
 South      86816.88   1045.987       0.106        85
 West       68143.46    908.579       0.078        77
East      3450077.53    995.981       0.099      3665
North     3494357.16   1017.576       0.113      3602
South     3370644.64   1002.571       0.105      3541
West      3394291.07    998.321       0.106      3565
east       102348.29   1149.981       0.174        92
north       66575.98   1091.410       0.169        65
south       87417.99   1150.237       0.111        81
west        73461.20   1113.048       0.101        69


### 🔹 Dimension-Preserving Group `.transform()`
Calculates each customer's mean transaction amount and computes individual transaction deviation z-scores. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.groupby('customer_id')['transaction_amount'].transform('mean')`


In [5]:
df_sub = df.dropna(subset=['transaction_amount']).head(100).copy()
df_sub['cust_avg_amt'] = df_sub.groupby('customer_id')['transaction_amount'].transform('mean')
df_sub['amt_ratio'] = df_sub['transaction_amount'] / df_sub['cust_avg_amt']
print(df_sub[['transaction_id', 'customer_id', 'transaction_amount', 'cust_avg_amt', 'amt_ratio']].head(5))

  transaction_id customer_id  transaction_amount  cust_avg_amt  amt_ratio
0       TX110686      C82845             1216.33       1216.33     1.0000
1       TX107170      C85674              324.99        324.99     1.0000
2       TX108328      C32431              136.66        922.13     0.1482
3       TX108563      C54057              124.21        124.21     1.0000
4       TX107002      C95649             1284.68       1284.68     1.0000


### 🔹 Group Filtering with `.filter()`
Extracts all transactions from high-volume customer accounts (customers with > 5 transactions). Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.groupby('customer_id').filter(lambda g: len(g) >= 5)`


In [6]:
high_vol_cust = df.groupby('customer_id').filter(lambda g: len(g) >= 5)
print(f'Transactions from High-Volume Customers: {len(high_vol_cust)}')
print(high_vol_cust[['transaction_id', 'customer_id', 'transaction_amount']].head(3))

Transactions from High-Volume Customers: 14824
  transaction_id customer_id  transaction_amount
0       TX110686      C82845             1216.33
1       TX107170      C85674              324.99
2       TX108328      C32431              136.66


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Identifying Outlier Transactions with Group Z-Scores

**Approach:** Flag transactions that are 3 standard deviations above the customer's average spend.
**Syntax:** `(df['amt'] - df.groupby('cust')['amt'].transform('mean')) / df.groupby('cust')['amt'].transform('std') > 3`


In [7]:
cust_means = df.groupby('customer_id')['transaction_amount'].transform('mean')
cust_stds = df.groupby('customer_id')['transaction_amount'].transform('std').fillna(1.0)
df_flagged = df.assign(z_score=(df['transaction_amount'] - cust_means) / cust_stds)
outliers = df_flagged[df_flagged['z_score'] > 2.5]
print(f'Found {len(outliers)} statistical outlier transactions!')
print(outliers[['transaction_id', 'customer_id', 'transaction_amount', 'z_score']].head(3))

Found 3 statistical outlier transactions!
      transaction_id customer_id  transaction_amount   z_score
1240        TX103625      C34356             1939.96  2.603843
4258        TX102917      C48427             1993.80  2.541286
13501       TX101204      C80912             1951.69  2.526769
